In [1]:
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import json
Dir_github = '/work/desai-lab/xuanyang/Project/ABC/github'
Dir_dataset = '/work/desai-lab/ABC_dataset'
Dir_preprocess = os.path.join(Dir_dataset,'derivatives','fmriprep')
project = 'ABC_valence'

In [2]:
def get_timing(df_eprime):
    List_vars = ['subid','visit','trial_onset','trial_RT','trial_btnprs','condition','correct']
    df = df_eprime.copy()

    df['onset'] = df['beginReal.RTTime']
    df.rename({'Subject':'subid','Session':'visit','Procedure':'condition'},axis=1,inplace=True)
    df.loc[df['condition']=='presentWordFam','condition'] = 'word'
    df.loc[df['condition']=='presentPic','condition'] = 'picture'

    for var,var_word,var_pic in zip(['trial_onset','trial_btnprs','trial_RT'],
                                    ['famPresent.OnsetTime','famPresent.RESP','famPresent.RTTime'],
                                    ['famPresentPic.OnsetTime','famPresentPic.RESP','famPresentPic.RTTime']):
        df[var] = df[var_word].values
        df.loc[df[var].isna(),var] = df.loc[df[var].isna(),var_pic].values

    df['trial_onset'] = (df['trial_onset'].values - df['onset'].values)/1000
    df['trial_RT'] = (df['trial_RT'].values - df['onset'].values)/1000

    df = df.loc[df['condition'].isin(['word','picture']),List_vars].reset_index(drop=True)

    return(df)


In [3]:
df_master = pd.read_csv(os.path.join(Dir_github,'derivatives','test_Fam','df_master_Fam120.csv'))
dfs_eprime = pd.read_csv(os.path.join(Dir_github,'derivatives','test_Fam','dfs_eprime_Fam_by1361.csv'))
df_master

,subid,visit,Fam_fmri,Fam_eprime,age,sex,trans_x,trans_y,trans_z,rot_x,rot_y,rot_z,motion_max
0,1005,1.0,1.0,1.0,65.0,2.0,0.041617,0.267869,0.463099,0.007059,0.003916,0.003123,0.463099
1,1007,1.0,1.0,1.0,69.0,2.0,0.383059,0.671302,0.386057,0.009304,0.001248,0.003724,0.671302
2,1008,1.0,1.0,1.0,77.0,2.0,0.075486,0.203145,0.118612,0.005849,0.000214,0.001656,0.203145
3,1009,1.0,1.0,1.0,75.0,2.0,0.000065,0.349682,0.215196,0.003958,0.000177,0.000196,0.349682
4,1018,1.0,1.0,1.0,60.0,2.0,0.026452,0.412780,0.433625,0.003764,0.005943,0.000393,0.433625
...,...,...,...,...,...,...,...,...,...,...,...,...,...
115,1302,1.0,1.0,1.0,23.0,1.0,0.565818,0.318678,0.295166,0.004322,0.001432,0.012690,0.565818
116,1305,1.0,1.0,1.0,61.0,2.0,0.061026,0.375817,0.324899,0.005508,0.000556,0.000923,0.375817
117,1306,1.0,1.0,1.0,28.0,2.0,0.485715,0.411004,0.707648,0.018751,0.001389,0.016267,0.707648
118,1311,1.0,1.0,1.0,20.0,2.0,0.174499,0.285192,0.111143,0.003649,0.008804,0.005942,0.285192


# Pipeline

In [21]:
nWM = 7
nCSF = 7
nPolort = 2
nMotion = 6
censor = 1

contrast = 'BtnPrs'
pipeline_clean = f"test_Fam_mot{nMotion}csf{nCSF}wm{nWM}plt{nPolort}cen{censor}"
pipeline = f"{pipeline_clean}_{contrast}"

Dir_clean = os.path.join(Dir_github,'derivatives',pipeline_clean)
Dir_pipeline = os.path.join(Dir_github,'derivatives',pipeline)
if not os.path.exists(Dir_pipeline):
    os.makedirs(Dir_pipeline)
    os.makedirs(os.path.join(Dir_pipeline,'scripts'))
pipeline

'test_Fam_mot6csf7wm7plt2cen1_BtnPrs'

In [22]:

for i, row in df_master.iterrows():


    subid = int(row['subid'])
    visit = int(row['visit'])
    Dir_subject = os.path.join(Dir_clean,'results',f"sub-{subid}",f"sub-{subid}.results")
    Dir_output = os.path.join(Dir_pipeline,'results','firstlevel',f'sub-{subid}')
    if not os.path.exists(Dir_output):
        os.makedirs(Dir_output)


    ### timing
    df_eprime = dfs_eprime.loc[dfs_eprime['Subject']==subid].loc[dfs_eprime['Session']==visit].reset_index(drop=True)
    df = get_timing(df_eprime)
    df.to_csv(os.path.join(Dir_output,f"sub-{subid}_task-Fam_timing.csv"),index=False)

    df_master.loc[i,'#trials'] = df.shape[0]
    df_master.loc[i,'#trials_btnprs'] = df.dropna().shape[0]

    # word and picture

    List_values = df.loc[df['condition']=='word','trial_onset'].astype(str).values
    path_word = os.path.join(Dir_output,f"sub-{subid}_task-Fam_timing-word.1D")
    with open(path_word,'w') as f:
        f.writelines('\n'.join(List_values))

    List_values = df.loc[df['condition']=='picture','trial_onset'].astype(str).values
    path_picture = os.path.join(Dir_output,f"sub-{subid}_task-Fam_timing-picture.1D")
    with open(path_picture,'w') as f:
        f.writelines('\n'.join(List_values))

    # button press
    List_values = df.loc[~df['trial_btnprs'].isna(),'trial_RT'].astype(str).values
    path_btnprs = os.path.join(Dir_output,f"sub-{subid}_task-Fam_timing-btnprs.1D")
    with open(path_btnprs,'w') as f:
        f.writelines('\n'.join(List_values))

    ### first-level analysis
    # T1 mask
    mask = os.path.join(Dir_subject,f'full_mask.sub-{subid}+tlrc.BRIK.gz')

    # fmri
    img = os.path.join(Dir_subject,f"errts.sub-{subid}.tproject+tlrc.BRIK")

    # motion censor
    path_motion_outlier = os.path.join(Dir_clean,'results',f"sub-{subid}",f'sub-{subid}_ses-visit0{visit}_task-Fam_motionsCensor.1D')

    # afni_proc.py script
    scriptFile = os.path.join(Dir_output,f'firstlevel_{subid}.sh')

    with open(scriptFile, "w") as f:
        f.write("#!/bin/tcsh\n")
        f.write(f"/work/apps/AFNI/linux_openmp_64/afni_proc.py -subj_id sub-{subid} \\\n")
        f.write("-scr_overwrite -blocks mask regress \\\n")
        f.write(f"-dsets {img} \\\n")
        
        f.write(f"-mask_import T1 {mask} \\\n")
        f.write(f"-mask_apply epi \\\n")
        f.write(f"-regress_stim_times {path_word} {path_picture} {path_btnprs} \\\n")
        f.write("-regress_basis 'SPMG1' \\\n")
        f.write(f"-regress_stim_types times times times \\\n")
        f.write("-regress_stim_labels word pic btn \\\n")
        f.write("-regress_opts_3dD -jobs 6 \\\n")
        f.write(f"-gltsym 'SYM: word -pic' -glt_label 1 word_pic \\\n")
        f.write("-regress_polort -1 \\\n")
        f.write("-regress_3dD_stop \\\n")
        f.write("-regress_reml_exec \\\n")
        f.write("-regress_make_ideal_sum sum_ideal.1D \\\n")
        f.write("-regress_run_clustsim no \\\n")
        f.write("\n")

    masterScript = os.path.join(Dir_pipeline,'scripts',f'run_firstlevel_{subid}.sh')
    with open(masterScript, "w") as f:
        f.write("#!/bin/tcsh\n")
        f.write(f"cd {os.path.join(Dir_output)} \n")
        f.write(f"tcsh {scriptFile} \n")
        f.write(f"tcsh proc.sub-{subid} \n")
        f.write(f"rm {Dir_output}/sub-{subid}.results/pb00.* \n")
        f.write(f"rm {Dir_output}/sub-{subid}.results/all_runs.* \n")
        f.write(f"rm {Dir_output}/sub-{subid}.results/fitts.* \n")
        f.write(f"rm {Dir_output}/sub-{subid}.results/errts.* \n")
    print(subid,'generated correctly.')


1005 generated correctly.
1007 generated correctly.
1008 generated correctly.
1009 generated correctly.
1018 generated correctly.
1019 generated correctly.
1020 generated correctly.
1021 generated correctly.
1024 generated correctly.
1025 generated correctly.
1030 generated correctly.
1032 generated correctly.
1037 generated correctly.
1041 generated correctly.
1042 generated correctly.
1047 generated correctly.
1048 generated correctly.
1049 generated correctly.
1051 generated correctly.
1052 generated correctly.
1053 generated correctly.
1055 generated correctly.
1056 generated correctly.
1060 generated correctly.
1062 generated correctly.
1065 generated correctly.
1069 generated correctly.
1070 generated correctly.
1072 generated correctly.
1073 generated correctly.
1076 generated correctly.
1078 generated correctly.
1079 generated correctly.
1081 generated correctly.
1083 generated correctly.
1085 generated correctly.
1086 generated correctly.
1087 generated correctly.
1091 generat

In [23]:
Dir_logs = os.path.join(Dir_pipeline,'logs')
if not os.path.exists(Dir_logs):
    os.makedirs(Dir_logs)

jobname = pipeline
ncores = 6
partition = 'desailab-48core'
script_slurm = os.path.join(Dir_pipeline,'scripts','Batch_slurm_firstlevel.sh')
array = '1001-1323'
#array = '001-013'
with open(script_slurm, "w") as f:
    f.write("#!/bin/sh\n")
    f.write("### name your job \n")
    f.write(f"#SBATCH --job-name={jobname} \n")
    f.write("### specify number of cores to use \n")
    f.write(f"#SBATCH -n {ncores} \n")
    f.write("#SBATCH -N 1 \n".format(ncores))
    f.write("### which partition do you want to use? \n")
    f.write(f"#SBATCH -p {partition} \n")
    f.write("### %a specifies that the output file for each subject will be named according to their array ID \n")
    f.write("# Where to output log files? \n")
    f.write(f"#SBATCH --output='{Dir_logs}/{pipeline}-%A_%a.log' \n")
    f.write(f"#SBATCH --error='{Dir_logs}/{pipeline}-%A_%a.err' \n")
    f.write("### list the subjects that you want to submit in this array \n")
    f.write(f"#SBATCH --array={array} \n")
    f.write("module load afni \n")
    f.write("module load python3/anaconda/2021.11 \n")
    f.write('echo "Slurm job ID: " $SLURM_JOB_ID \n')
    f.write('echo "Slurm array task ID: " $SLURM_ARRAY_TASK_ID \n')

    f.write('printf -v subj "%03d" $SLURM_ARRAY_TASK_ID \n')
    f.write('echo "Running 3dTproject on sub-$subj" \n')
    f.write(f"cd {os.path.join(Dir_pipeline,'scripts')} \n")
    f.write('tcsh run_firstlevel_$subj.sh \n')
    f.write('echo "Finished firstlevel sub-$subj" \n')
    f.write('date \n')
    